# Demo — Lectura del DW Local (PostgreSQL)

Este notebook demuestra que el backup del Data Warehouse Gold está operativo en PostgreSQL local.

**Prerequisitos:**
```bash
pip install psycopg[binary] pandas python-dotenv
docker compose up -d postgres   # desde la raíz del repo
```

In [0]:
import os
from pathlib import Path
import psycopg
import pandas as pd
from dotenv import load_dotenv

# Lee credenciales desde dw-backup/.env — nunca se suben a git
load_dotenv(Path(__file__).parent / '.env' if '__file__' in globals() else Path('.env'))

CONN = dict(
    host='localhost',
    port=int(os.environ['PG_PORT']),
    dbname=os.environ['PG_DB'],
    user=os.environ['PG_USER'],
    password=os.environ['PG_PASSWORD'],
)

conn = psycopg.connect(**CONN)
print('Conexión OK —', conn.info.server_version)

## 1. Conteo de filas por tabla

In [0]:
tablas = [
    'dim_causa', 'dim_demografia', 'dim_fuente',
    'dim_geografia', 'dim_tiempo',
    'fact_defunciones', 'fact_morbilidad',
]

counts = []
with conn.cursor() as cur:
    for t in tablas:
        cur.execute(f'SELECT COUNT(*) FROM gold_ss2.{t}')
        counts.append({'tabla': t, 'filas': cur.fetchone()[0]})

pd.DataFrame(counts).set_index('tabla')

## 2. Defunciones por año y departamento (Guatemala)

In [0]:
SQL_DEFUNCIONES = """
SELECT
    d.anio,
    d.periodo_covid,
    g.departamento,
    SUM(f.total_defunciones) AS defunciones
FROM gold_ss2.fact_defunciones f
JOIN gold_ss2.dim_tiempo     d ON f.tiempo_sk    = d.tiempo_sk
JOIN gold_ss2.dim_geografia  g ON f.geografia_sk = g.geografia_sk
WHERE g.nivel_geo = 'departamento'
  AND g.pais      = 'Guatemala'
GROUP BY d.anio, d.periodo_covid, g.departamento
ORDER BY d.anio, defunciones DESC
"""

df_def = pd.read_sql(SQL_DEFUNCIONES, conn)
print(f'{len(df_def)} filas')
df_def.head(20)

## 3. Top 10 causas de morbilidad (período COVID vs pre-COVID)

In [0]:
SQL_MORBILIDAD = """
SELECT
    t.periodo_covid,
    c.codigo_origen,
    c.descripcion,
    SUM(f.casos_total) AS casos
FROM gold_ss2.fact_morbilidad f
JOIN gold_ss2.dim_tiempo  t ON f.tiempo_sk = t.tiempo_sk
JOIN gold_ss2.dim_causa   c ON f.causa_sk  = c.causa_sk
GROUP BY t.periodo_covid, c.codigo_origen, c.descripcion
ORDER BY t.periodo_covid, casos DESC
LIMIT 20
"""

df_morb = pd.read_sql(SQL_MORBILIDAD, conn)
df_morb

## 4. Mortalidad WHO — Guatemala vs Costa Rica (tasa x 100k)

In [0]:
SQL_WHO = """
SELECT
    g.pais,
    t.anio,
    ROUND(AVG(f.tasa_mortalidad_x100k)::numeric, 2) AS tasa_promedio_x100k
FROM gold_ss2.fact_defunciones f
JOIN gold_ss2.dim_tiempo    t ON f.tiempo_sk    = t.tiempo_sk
JOIN gold_ss2.dim_geografia g ON f.geografia_sk = g.geografia_sk
WHERE f.tasa_mortalidad_x100k IS NOT NULL
  AND g.nivel_geo = 'pais'
GROUP BY g.pais, t.anio
ORDER BY g.pais, t.anio
"""

df_who = pd.read_sql(SQL_WHO, conn)
df_who

In [0]:
conn.close()
print('Conexión cerrada — demo completo.')